In [ ]:
import carla

In [ ]:
CARLA_HOST = "localhost"
CARLA_PORT = 2000
TRAFFIC_MANAGER_PORT = 8000
CARLA_TIMEOUT = 10.0
FOLLOWING_DISTANCE = 3.0
SPEED_DIFFERENCE_PERCENT = 10.0

In [ ]:
def connect_to_carla() -> tuple[carla.Client, carla.World]:
    '''Create a carla CLIENT and return the active world'''

    client = carla.Client(
        CARLA_HOST,
        CARLA_PORT
    )

    client.set_timeout(CARLA_TIMEOUT)

    world = client.get_world()

    if not world:
        raise RuntimeError(
            "Carla failed to return active world"
        )
    
    return client, world


In [ ]:
def get_vehicle_blueprint(world: carla.World) -> carla.BlueprintLibrary:

    bp_lib = world.get_blueprint_library().filter("vehicle.*")

    if not bp_lib:
        raise RuntimeError(
            "Carla failed to return a valid list of blueprints"
        )

    return bp_lib[0]

In [ ]:
def get_spawn_points(world: carla.World) -> carla.Transform:

    spawn_points = world.get_map().get_spawn_points()

    if not spawn_points:
        raise RuntimeError(
            "Carla failed to return a valid list of possible spawn points."
        )

    return spawn_points[0]

In [ ]:
def spawn_vehicle(world: carla.World, spawn_point: carla.Transform, vehicle_bp: carla.ActorBlueprint) -> carla.Vehicle:
    '''Spawn the decided vehicle at the decided point'''

    vehicle_bp = get_vehicle_blueprint(world)
    spawn_point = get_spawn_points(world)
    vehicle = world.try_spawn_actor(
        vehicle_bp,
        spawn_point
    )

    if not vehicle:
        raise RuntimeError(
            "Carla failed to spawn the vehicle."
        )

    return vehicle


In [ ]:
def configure_traffic_manager(traffic_manager: carla.TrafficManager) -> None:

    traffic_manager.set_global_distance_to_leading_vehicle(
        FOLLOWING_DISTANCE
    )

    traffic_manager.global_percentage_speed_difference(
        FOLLOWING_DISTANCE
    )


In [ ]:
def enable_autopilot(vehicle: carla.Vehicle, traffic_manager_port: int) -> None:

    vehicle.set_autopilot(
        True, 
        traffic_manager_port
    )

In [ ]:
def main() -> None:

    client, world = connect_to_carla()

    vehicle_bp = get_vehicle_blueprint(world)

    spawn_point = get_spawn_points(world)

    traffic_manager = client.get_trafficmanager(
        TRAFFIC_MANAGER_PORT
    )

    configure_traffic_manager(
        traffic_manager
    )

    vehicle = spawn_vehicle(
        world,
        spawn_point,
        vehicle_bp
    )

    try:
        enable_autopilot(
            vehicle,
            TRAFFIC_MANAGER_PORT
        )

        print("=== TRAFFIC MANAGER ===")
        print(f"Vehicle ID: {vehicle.id}")
        print(f"Vehicle type: {vehicle.type_id}")
        print()
        print(
            f"Following distance: "
            f"{FOLLOWING_DISTANCE} m"
        )
        print(
            f"Speed difference: "
            f"{FOLLOWING_DISTANCE}%"
        )
        print()
        print(
            "Vehicle is now controlled by "
            "the Traffic Manager."
        )

        input(
            "\nPress ENTER to stop the simulation..."
        )

    finally:
        if vehicle.is_alive:
            vehicle.set_autopilot(False)
            vehicle.destroy()
            print("Vehicle destroyed")





In [ ]:
if __name__ == "__main__":
    main()